In [59]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [60]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [61]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)
ERROR: No matching distribution found for random


In [62]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import psycopg
from psycopg import sql

In [65]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [66]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:

            with conn.cursor() as cur:

                # Check whether target database already exists
                cur.execute(
                    """
                    SELECT EXISTS (
                        SELECT 1
                        FROM pg_database
                        WHERE datname = %s
                    );
                    """,
                    (db_name,),
                )

                exists = cur.fetchone()[0]

                if exists:
                    print(f"Database '{db_name}' already exists.")

                else:
                    query = sql.SQL("CREATE DATABASE {}").format(
                        sql.Identifier(db_name)
                    )

                    cur.execute(query)

                    print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [70]:
create_db_meta("meta")

Database 'meta' already exists.


In [71]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:

                # Check if table already exists
                cur.execute("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM information_schema.tables
                        WHERE table_schema = 'public'
                        AND table_name = 'metadata'
                    );
                    """)

                exists = cur.fetchone()[0]

                if exists:
                    print("metadata table already exists.")

                else:
                    cur.execute("""
                        CREATE TABLE metadata (
                            site_name VARCHAR(100) NOT NULL,
                            latitude DOUBLE PRECISION NOT NULL,
                            longitude DOUBLE PRECISION NOT NULL
                        );
                        """)

                    print("metadata table created.")

            conn.commit()

    except psycopg.Error as e:
        print(e)

In [72]:
create_table_meta()

metadata table already exists.


In [75]:
def insert_sites():

    with psycopg.connect(
        dbname="meta",
        user="postgres",
        password="123789",
        host="localhost",
        port="5000",
    ) as conn:

        with conn.cursor() as cur:

            # Check existing number of sites
            cur.execute("""
                SELECT COUNT(*)
                FROM metadata;
                """)

            existing_sites = cur.fetchone()[0]

            # Already 10,000 sites
            if existing_sites >= 10000:
                print(
                    f"{existing_sites} sites already exist. " "No new sites inserted."
                )
                return

            # Generate sites only if needed
            sites_df = generate_site_data(10000)

            for _, row in sites_df.iterrows():

                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)
                    """,
                    (
                        row["site_code"],
                        row["latitude"],
                        row["longitude"],
                    ),
                )

        conn.commit()

    print("Sites inserted successfully!")

In [76]:
insert_sites()

10000 sites already exist. No new sites inserted.


In [77]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [78]:
data = get_sites()

In [79]:
type(data[0])

tuple

In [80]:
len(data)

10000

In [81]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:

            with conn.cursor() as cur:

                # Check whether target database already exists
                cur.execute(
                    """
                    SELECT EXISTS (
                        SELECT 1
                        FROM pg_database
                        WHERE datname = %s
                    );
                    """,
                    (db_name,),
                )

                exists = cur.fetchone()[0]

                if exists:
                    print(f"Database '{db_name}' already exists.")

                else:
                    query = sql.SQL("CREATE DATABASE {}").format(
                        sql.Identifier(db_name)
                    )

                    cur.execute(query)

                    print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [82]:
create_db_site_weather("site_weather")

Database 'site_weather' already exists.


In [85]:
def create_table_site_weather():
    try:
        with psycopg.connect(
            dbname="site_weather",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:

                # Check if table already exists
                cur.execute("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM information_schema.tables
                        WHERE table_schema = 'public'
                        AND table_name = 'metadata'
                    );
                    """)

                exists = cur.fetchone()[0]

                if exists:
                    print("metadata table already exists.")

                else:
                    cur.execute("""
                        CREATE TABLE metadata (
                            site_name VARCHAR(100) NOT NULL,
                            latitude DOUBLE PRECISION NOT NULL,
                            longitude DOUBLE PRECISION NOT NULL
                        );
                        """)

                    print("metadata table created.")

            conn.commit()

    except psycopg.Error as e:
        print(e)

In [86]:
create_table_site_weather()

metadata table created.


In [ ]:
import time
import datetime
import requests
import psycopg


DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

DB_NAME = "site_weather"

API_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

BATCH_SIZE = 100

REQUEST_DELAY = 0.1

RETRY_WAIT = 5

RATE_LIMIT_WAIT = 60

REQUEST_TIMEOUT = 120

start_datetime = datetime.datetime.fromisoformat(START_DATE)

end_datetime = datetime.datetime.fromisoformat(END_DATE) + datetime.timedelta(days=1)

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute(
            """
            SELECT site_name
            FROM site_weather
            WHERE time_interval >= %s
              AND time_interval < %s
            GROUP BY site_name
            HAVING COUNT(*) = 24;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        completed_sites = {row[0] for row in cur.fetchall()}

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:

        pending_sites.append(
            (
                site_name,
                latitude,
                longitude,
            )
        )

print("=" * 60)
print("RESUME CHECK")
print("=" * 60)

print(f"Total sites       : {len(data)}")
print(f"Completed sites   : {len(completed_sites)}")
print(f"Remaining sites   : {len(pending_sites)}")

print("=" * 60)

DATE RANGE
Requested date : 2026-08-11
SQL start      : 2026-08-11 00:00:00
SQL end        : 2026-08-12 00:00:00
RESUME CHECK
Total sites       : 10000
Completed sites   : 0
Remaining sites   : 10000


In [ ]:
def extract_site_weather(site_name, weather):
    
    if weather.get("error"):

        reason = weather.get("reason", "Unknown API error")

        raise ValueError(f"API error for {site_name}: {reason}")
    
    hourly = weather.get("hourly")

    if not hourly:
        raise ValueError(f"No hourly data for {site_name}")

    times = hourly.get("time", [])

    temperatures = hourly.get("temperature_2m", [])

    humidity = hourly.get("relative_humidity_2m", [])

    radiation = hourly.get("direct_radiation", [])

    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
    ):

        raise ValueError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )


    rows = []

    for i in range(24):
        
        timestamp = datetime.datetime.fromisoformat(times[i])

        if timestamp.tzinfo is not None:

            timestamp = timestamp.replace(tzinfo=None)

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows

# Fetches a batch of sites from the API and handles retries rates
def fetch_batch(batch):

    latitudes = [str(site[1]) for site in batch]

    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            [
                "temperature_2m",
                "relative_humidity_2m",
                "direct_radiation",
            ]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    while True:
        
        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )
            
            if response.status_code == 200:

                result = response.json()

                if isinstance(result, dict):

                    result = [result]

                if not isinstance(result, list):

                    raise ValueError("Invalid API response format.")

                if len(result) != len(batch):

                    raise ValueError(
                        f"Expected "
                        f"{len(batch)} locations, "
                        f"but API returned "
                        f"{len(result)}."
                    )

                # One message for the WHOLE batch.
                print(f"API SUCCESS | " f"Batch contains {len(batch)} sites")

                return result

            if response.status_code == 429:

                print(f"API RATE LIMIT | " f"Waiting {RATE_LIMIT_WAIT} seconds...")

                time.sleep(RATE_LIMIT_WAIT)

                continue
            
            if response.status_code >= 500:

                print(
                    f"API SERVER ERROR "
                    f"{response.status_code} | "
                    f"Retrying in {RETRY_WAIT} seconds..."
                )

                time.sleep(RETRY_WAIT)

                continue

            print(
                f"API ERROR "
                f"{response.status_code} | "
                f"Retrying in {RETRY_WAIT} seconds..."
            )

            time.sleep(RETRY_WAIT)

        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"NETWORK ERROR | {e}")
            print(f"Retrying in {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)

        except requests.HTTPError as e:
            print(f"HTTP ERROR | {e}")
            print(f"Retrying in {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)

        except ValueError as e:
            print(f"DATA ERROR | {e}")

# Used ONLY if a site inside a successful batch has
# an invalid/error response.

def fetch_single_site(site):

    while True:
        
        try:
            
            result = fetch_batch([site])

            weather = result[0]
            
            extract_site_weather(
                site[0],
                weather,
            )

            return weather

        except Exception as e:

            print(f"SITE ERROR | " f"{site[0]} | {e}")

            print(f"Retrying site in " f"{RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)

total_rows = 0

total_pending = len(pending_sites)


if total_pending == 0:
    
    print("All sites are already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE

    with psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(
                    0,
                    total_pending,
                    BATCH_SIZE,
                ),
                start=1,
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                print()
                print("=" * 60)
                print(f"BATCH " f"{batch_number}/" f"{total_batches}")

                print(f"Sites " f"{start + 1} - " f"{start + len(batch)}")

                print("=" * 60)
                
                api_results = fetch_batch(batch)

                batch_rows = []

                successful_sites = 0

                for i, site in enumerate(batch):

                    site_name = site[0]

                    weather = api_results[i]

                    try:

                        rows = extract_site_weather(
                            site_name,
                            weather,
                        )

                        batch_rows.extend(rows)

                        successful_sites += 1

                    except Exception as e:
                        print(f"SITE RESPONSE ERROR | " f"{site_name} | {e}")
                        print(f"Retrying " f"{site_name} individually...")

                        weather = fetch_single_site(site)

                        rows = extract_site_weather(
                            site_name,
                            weather,
                        )

                        batch_rows.extend(rows)

                        successful_sites += 1

                if successful_sites != len(batch):

                    raise RuntimeError(
                        f"Batch validation failed. "
                        f"Expected {len(batch)} sites, "
                        f"validated {successful_sites}."
                    )

                cur.executemany(
                    """
                    INSERT INTO site_weather (
                        site_name,
                        time_interval,
                        temperature,
                        humidity,
                        solar_radiance
                    )
                    VALUES (
                        %s,
                        %s,
                        %s,
                        %s,
                        %s
                    )
                    ON CONFLICT (
                        site_name,
                        time_interval
                    )
                    DO NOTHING;
                    """,
                    batch_rows,
                )

                conn.commit()

                total_rows += len(batch_rows)
                print(
                    f"BATCH SAVED | "
                    f"{len(batch)} sites | "
                    f"{len(batch_rows)} hourly rows | "
                    f"Committed"
                )

                time.sleep(REQUEST_DELAY)


print()
print("=" * 60)
print("BATCH PROCESSING FINISHED")
print("=" * 60)

print(f"Rows processed in this run: " f"{total_rows}")

print("=" * 60)


BATCH 1/100
Sites 1 - 100
API RATE LIMIT | Waiting 60 seconds...


In [58]:
# ============================================================
# FINAL DATABASE VERIFICATION
# ============================================================

with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        # ====================================================
        # COMPLETED SITE COUNT
        # ====================================================

        cur.execute(
            """
            SELECT COUNT(*)
            FROM (
                SELECT site_name
                FROM site_weather
                WHERE time_interval >= %s
                  AND time_interval < %s
                GROUP BY site_name
                HAVING COUNT(*) = 24
            ) AS completed;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        actual_sites = cur.fetchone()[0]

        # ====================================================
        # TOTAL ROW COUNT
        # ====================================================

        cur.execute(
            """
            SELECT COUNT(*)
            FROM site_weather
            WHERE time_interval >= %s
              AND time_interval < %s;
            """,
            (
                start_datetime,
                end_datetime,
            ),
        )

        actual_rows = cur.fetchone()[0]


# ============================================================
# EXPECTED VALUES
# ============================================================

expected_sites = len(data)

expected_rows = expected_sites * 24


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print(f"Expected sites : " f"{expected_sites}")

print(f"Actual sites   : " f"{actual_sites}")

print(f"Expected rows  : " f"{expected_rows}")

print(f"Actual rows    : " f"{actual_rows}")

print("=" * 60)


if actual_sites == expected_sites and actual_rows == expected_rows:

    print("SUCCESS: All sites and " "all 24 hourly records are complete.")

else:

    print("NOT COMPLETE YET.")

    print(f"Missing rows: " f"{expected_rows - actual_rows}")

    print(f"Missing sites: " f"{expected_sites - actual_sites}")


FINAL VERIFICATION
Expected sites : 10000
Actual sites   : 10000
Expected rows  : 240000
Actual rows    : 240000
SUCCESS: All sites and all 24 hourly records are complete.
